In [1]:
import requests
import pandas as pd

In [2]:
API_KEY = "e866cce851a1433995a56169f0420e64"
Base_Url = 'https://api.football-data.org/v4/'
headers = {
    "X-Auth-Token":API_KEY
}


In [3]:
def fetchMatchData(competitionId, season):
    url = f'{Base_Url}competitions/{competitionId}/matches?season={season}'
    response = requests.get(url=url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        matches = data['matches']
        return pd.DataFrame(matches)
    else:
        print(f'Failed to fetch data: {response.status_code}')
        return None


In [4]:
competitionId = 'PL'
season= '2023'

matchData = fetchMatchData(competitionId=competitionId, season=season)
if matchData is not None:
    matchData.head()

In [5]:
matchData.tail()

,area,competition,season,id,utcDate,status,matchday,stage,group,lastUpdated,homeTeam,awayTeam,score,odds,referees
375,"{'id': 2072, 'name': 'England', 'code': 'ENG',...","{'id': 2021, 'name': 'Premier League', 'code':...","{'id': 1564, 'startDate': '2023-08-11', 'endDa...",436318,2024-05-19T15:00:00Z,FINISHED,38,REGULAR_SEASON,None,2024-05-19T20:21:04Z,"{'id': 354, 'name': 'Crystal Palace FC', 'shor...","{'id': 58, 'name': 'Aston Villa FC', 'shortNam...","{'winner': 'HOME_TEAM', 'duration': 'REGULAR',...",{'msg': 'Activate Odds-Package in User-Panel t...,"[{'id': 11317, 'name': 'Darren Bond', 'type': ..."
376,"{'id': 2072, 'name': 'England', 'code': 'ENG',...","{'id': 2021, 'name': 'Premier League', 'code':...","{'id': 1564, 'startDate': '2023-08-11', 'endDa...",436319,2024-05-19T15:00:00Z,FINISHED,38,REGULAR_SEASON,None,2024-06-02T20:20:54Z,"{'id': 64, 'name': 'Liverpool FC', 'shortName'...","{'id': 76, 'name': 'Wolverhampton Wanderers FC...","{'winner': 'HOME_TEAM', 'duration': 'REGULAR',...",{'msg': 'Activate Odds-Package in User-Panel t...,"[{'id': 11443, 'name': 'Chris Kavanagh', 'type..."
377,"{'id': 2072, 'name': 'England', 'code': 'ENG',...","{'id': 2021, 'name': 'Premier League', 'code':...","{'id': 1564, 'startDate': '2023-08-11', 'endDa...",436320,2024-05-19T15:00:00Z,FINISHED,38,REGULAR_SEASON,None,2024-05-19T20:21:04Z,"{'id': 389, 'name': 'Luton Town FC', 'shortNam...","{'id': 63, 'name': 'Fulham FC', 'shortName': '...","{'winner': 'AWAY_TEAM', 'duration': 'REGULAR',...",{'msg': 'Activate Odds-Package in User-Panel t...,"[{'id': 11626, 'name': 'Matt Donohue', 'type':..."
378,"{'id': 2072, 'name': 'England', 'code': 'ENG',...","{'id': 2021, 'name': 'Premier League', 'code':...","{'id': 1564, 'startDate': '2023-08-11', 'endDa...",436321,2024-05-19T15:00:00Z,FINISHED,38,REGULAR_SEASON,None,2024-06-02T20:20:54Z,"{'id': 65, 'name': 'Manchester City FC', 'shor...","{'id': 563, 'name': 'West Ham United FC', 'sho...","{'winner': 'HOME_TEAM', 'duration': 'REGULAR',...",{'msg': 'Activate Odds-Package in User-Panel t...,"[{'id': 11327, 'name': 'John Brooks', 'type': ..."
379,"{'id': 2072, 'name': 'England', 'code': 'ENG',...","{'id': 2021, 'name': 'Premier League', 'code':...","{'id': 1564, 'startDate': '2023-08-11', 'endDa...",436322,2024-05-19T15:00:00Z,FINISHED,38,REGULAR_SEASON,None,2024-05-19T20:21:04Z,"{'id': 356, 'name': 'Sheffield United FC', 'sh...","{'id': 73, 'name': 'Tottenham Hotspur FC', 'sh...","{'winner': 'AWAY_TEAM', 'duration': 'REGULAR',...",{'msg': 'Activate Odds-Package in User-Panel t...,"[{'id': 193220, 'name': 'Bobby Madley', 'type'..."


In [6]:
# Flatten the nested columns
def flatten_columns(df):
    # Extract relevant columns
    df['home_team'] = df['homeTeam'].apply(lambda x: x['name'] if isinstance(x, dict) else None)
    df['away_team'] = df['awayTeam'].apply(lambda x: x['name'] if isinstance(x, dict) else None)
    df['home_score'] = df['score'].apply(lambda x: x['fullTime']['home'] if isinstance(x['fullTime'], dict) else None)
    df['away_score'] = df['score'].apply(lambda x: x['fullTime']['away'] if isinstance(x['fullTime'], dict) else None)
    df['winner'] = df['score'].apply(lambda x: x['winner'] if isinstance(x, dict) else None)

    # Drop original nested columns
    df = df.drop(columns=['homeTeam', 'awayTeam', 'score', 'area', 'competition', 'season', 'odds', 'referees', 'stage', 'group'])
    return df

In [7]:
df = flatten_columns(matchData)

In [8]:
df['utcDate'] = pd.to_datetime(df['utcDate'])
df.head()

,id,utcDate,status,matchday,lastUpdated,home_team,away_team,home_score,away_score,winner
0,435943,2023-08-11 19:00:00+00:00,FINISHED,1,2024-06-02T20:20:54Z,Burnley FC,Manchester City FC,0,3,AWAY_TEAM
1,435944,2023-08-12 12:00:00+00:00,FINISHED,1,2023-09-19T20:20:30Z,Arsenal FC,Nottingham Forest FC,2,1,HOME_TEAM
2,435945,2023-08-12 14:00:00+00:00,FINISHED,1,2023-09-19T20:20:30Z,AFC Bournemouth,West Ham United FC,1,1,DRAW
3,435946,2023-08-12 14:00:00+00:00,FINISHED,1,2023-09-19T20:20:30Z,Brighton & Hove Albion FC,Luton Town FC,4,1,HOME_TEAM
4,435947,2023-08-12 14:00:00+00:00,FINISHED,1,2023-09-19T20:20:30Z,Everton FC,Fulham FC,0,1,AWAY_TEAM


In [9]:
df.to_csv('premier_league_2023_cleaned.csv', index=False)
print('saved')

saved


In [10]:
def fetch_teams(competition_id):
    """Fetches teams participating in a given competition."""
    url = f'{Base_Url}competitions/{competition_id}/teams'
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        teams = data['teams']
        return pd.DataFrame(teams)
    else:
        print(f'Failed to fetch teams: {response.status_code}')
        return None

def fetch_players(team_id):
    """Fetches player data for a given team."""
    url = f'{Base_Url}teams/{team_id}'
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        players = data['squad']
        return pd.DataFrame(players)
    else:
        print(f'Failed to fetch players for team {team_id}: {response.status_code}')
        return None

In [11]:
# Fetch teams in the competition
competition_id = 'PL'  # Premier League
teams_data = fetch_teams(competition_id)

In [12]:
if teams_data is not None:
    all_players = pd.DataFrame()
    for _, team in teams_data.iterrows():
        team_id = team['id']
        team_players = fetch_players(team_id)
        
        if team_players is not None:
            team_players['team'] = team['name']  # Add team name to player data
            all_players = pd.concat([all_players, team_players], ignore_index=True)

    # Save the player data to a CSV file
    all_players.to_csv('premier_league_2023_players.csv', index=False)
    print('Player data saved successfully!')

Failed to fetch players for team 73: 429
Failed to fetch players for team 76: 429
Failed to fetch players for team 338: 429
Failed to fetch players for team 340: 429
Failed to fetch players for team 349: 429
Failed to fetch players for team 351: 429
Failed to fetch players for team 354: 429
Failed to fetch players for team 397: 429
Failed to fetch players for team 402: 429
Failed to fetch players for team 563: 429
Failed to fetch players for team 1044: 429
Player data saved successfully!


In [13]:
all_players.head()

,id,name,position,dateOfBirth,nationality,team
0,4832,David Raya,Goalkeeper,1995-09-15,Spain,Arsenal FC
1,5530,Aaron Ramsdale,Goalkeeper,1998-05-14,England,Arsenal FC
2,6154,Ben White,Defence,1997-10-08,England,Arsenal FC
3,7889,Oleksandr Zinchenko,Defence,1996-12-15,Ukraine,Arsenal FC
4,9034,Takehiro Tomiyasu,Defence,1998-11-05,Japan,Arsenal FC


In [31]:
all_players.iloc[1]

id                       5530
name           Aaron Ramsdale
position           Goalkeeper
dateOfBirth        1998-05-14
nationality           England
team               Arsenal FC
Name: 1, dtype: object

In [36]:
all_players.iloc[190]['name'][0]

'P'